# 📥 Telegram to Google Drive — Interactive Downloader

Welcome! This notebook provides a complete, user-friendly interface to download files, videos, and albums from Telegram directly into your Google Drive. It handles everything from private groups to album selections with a live, interactive dashboard.

> 🤖 **Credits:** The code, UI, and architecture for this notebook were written and engineered in collaboration with **DeepSeek AI**.

### ✨ Features
- **📂 Auto-Organized:** Automatically creates subfolders named `Sender Name - Date` to keep your Drive clean.
- **⚡ Live Speed & ETA:** Real-time speedometer, time elapsed, and estimated time remaining.
- **✅ Interactive Selection:** Fetch an album, uncheck the files you don't want, and download only what you need.
- **🛡️ Auto-Resume:** If Colab disconnects, it compares file sizes and skips files already in your Drive.
- **📊 Live Dashboard:** A real-time dashboard showing your total files, storage used, and recent download history.
- **🔐 Secure Session Reuse:** Saves your Telegram session safely inside your own Google Drive, so you only log in once.

---
### ⚠️ Important Disclaimers
* **Google Drive Quota:** The free tier of Google Drive is limited to **15 GB**. If you hit this limit, file transfers will fail.
* **Google Colab ToS:** Colab is designed for interactive coding, not as a 24/7 bulk file-transfer server. Avoid downloading massive amounts of data in a single run to prevent your session from being flagged.
* **Private Groups:** To download from a private group or channel, your Telegram account **must** already be a member of that group.
* **Copyright:** This tool is intended for personal backups. Do not use it to mass-mirror copyrighted material. Use responsibly.

## ⚙️ Step 0: Environment Setup

Run this cell first. It will automatically install the required Python dependencies, mount your Google Drive, and create the necessary folders for downloads and credentials.

*If Colab asks for permission to access your Google Drive, click **"Connect to Google Drive"** and allow it.*

In [ ]:
#@title ⚙️ RUN: Environment Setup { display-mode: "form" }

import subprocess, sys, os, shutil
import importlib.metadata
from IPython.display import HTML, display

THEME = """
<style>
.section{background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);
         border-radius:10px;padding:12px 20px;color:#fff;margin:16px 0 12px;
         display:flex;align-items:center;gap:14px;
         box-shadow:0 4px 12px rgba(102,126,234,.2);
         font-family:-apple-system,'Segoe UI',system-ui,sans-serif}
.section .num{background:rgba(255,255,255,.25);border-radius:8px;
              padding:5px 12px;font-weight:700;font-size:12px;letter-spacing:.5px}
.section .title{font-size:16px;font-weight:600}
.badge{padding:10px 14px;border-radius:8px;margin:8px 0;font-size:14px;
       font-weight:500;font-family:-apple-system,system-ui,sans-serif}
.badge.ok{background:#e8f5e9;color:#2e7d32;border-left:4px solid #2e7d32}
.badge.err{background:#ffebee;color:#c62828;border-left:4px solid #c62828}
.badge.warn{background:#fff3e0;color:#e65100;border-left:4px solid #e65100}
.badge.info{background:#e3f2fd;color:#1565c0;border-left:4px solid #1565c0}
code{background:#f5f5f5;padding:2px 6px;border-radius:4px;
     font-family:'SF Mono',Menlo,monospace;font-size:12px}
</style>"""
display(HTML(THEME))

def _badge(kind, text):
    return HTML(f"<div class='badge {kind}'>{text}</div>")

def section(n, t):
    display(HTML(f"<div class='section'><span class='num'>STEP {n}</span>"
                 f"<span class='title'>{t}</span></div>"))

# ── Install & Print Versions ─────────────────────────────
packages = ["telethon", "tqdm", "nest_asyncio", "ipywidgets"]

print("Checking dependencies...")
for pkg in packages:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--disable-pip-version-check", pkg],
            capture_output=True, text=True, timeout=180, check=True,
        )
        version = importlib.metadata.version(pkg)
        print(f"✅ Installed {pkg:<15} (v{version})")
    except Exception as e:
        print(f"❌ Failed to install {pkg}: {e}")

# ── Mount Drive ──────────────────────────────────────────
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

import nest_asyncio, json, re, asyncio
import ipywidgets as w
nest_asyncio.apply()

# ── Folder layout ────────────────────────────────────────
DRIVE_ROOT    = "/content/drive/MyDrive"
DOWNLOADS_DIR = os.path.join(DRIVE_ROOT, "TelegramToGDrive Downloads")
CREDS_DIR     = os.path.join(DRIVE_ROOT, "TelegramToGDrive Credentials")

os.makedirs(DOWNLOADS_DIR, exist_ok=True)
os.makedirs(CREDS_DIR, exist_ok=True)

# ── Auto-migrate old credentials from Drive root ─────────
OLD_FILES = {
    os.path.join(DRIVE_ROOT, "tg_session.txt"): os.path.join(CREDS_DIR, "tg_session.txt"),
    os.path.join(DRIVE_ROOT, "tg_api.json"):    os.path.join(CREDS_DIR, "tg_api.json"),
}
migrated = []
for old_path, new_path in OLD_FILES.items():
    if os.path.exists(old_path) and not os.path.exists(new_path):
        try:
            shutil.move(old_path, new_path)
            migrated.append(os.path.basename(old_path))
        except Exception:
            pass

# ── State ────────────────────────────────────────────────
STATE = {
    "tg":          None,
    "tg_verified": False,
    "drive_root":  DOWNLOADS_DIR,
    "creds_dir":   CREDS_DIR,
    "tmp_dir":     "/content/tmp",
}
os.makedirs(STATE["tmp_dir"], exist_ok=True)

if migrated:
    print(f"\nMoved old credentials into new folder: {', '.join(migrated)}")

print(f"\n✅ Ready.")
print(f"Downloads → {DOWNLOADS_DIR}")
print(f"Credentials → {CREDS_DIR}")

## 🔧 Step 1: Initialize Backend

This cell loads all the background logic, custom error handlers, and async loops.

**You don't need to change anything here.** Just click the play button (▶️) and wait for the green `✅ Backend ready.` badge to appear.

In [ ]:
#@title 🔧 RUN: Initialize Backend { display-mode: "form" }

if "STATE" not in globals():
    display(_badge("warn", "⚠️  Run Setup first."))
else:
    import os, re, json, asyncio, shutil, time, logging
    from telethon import TelegramClient
    from telethon.sessions import StringSession
    from telethon.errors import (
        SessionPasswordNeededError, FloodWaitError,
        ApiIdInvalidError, PhoneNumberInvalidError,
        PhoneCodeInvalidError, PhoneCodeExpiredError,
        PasswordHashInvalidError, UsernameNotOccupiedError,
        UsernameInvalidError, ChannelPrivateError, MessageIdInvalidError,
        AuthKeyUnregisteredError, TimedOutError, ServerError
    )

    logging.getLogger('telethon').setLevel(logging.ERROR)

    _LINK_RE = re.compile(r"https?://t\.me/(?:c/)?([^/?]+)(?:/\d+)?/(\d+)")

    def _friendly(e):
        n, m = type(e).__name__, str(e)
        M = {
            "AuthKeyUnregisteredError":  "Session invalidated — click 🔄 Restart login",
            "ApiIdInvalidError":         "API ID/hash invalid — check my.telegram.org/apps",
            "PhoneNumberInvalidError":   "Phone invalid — include country code",
            "PhoneCodeInvalidError":     "Wrong login code",
            "PhoneCodeExpiredError":     "Code expired — click 1. Send code again",
            "PasswordHashInvalidError":  "Wrong 2FA password",
            "SessionPasswordNeededError":"2FA enabled — enter password, click 3. Submit 2FA",
            "UsernameNotOccupiedError":  "Chat not found — join it first",
            "UsernameInvalidError":      "Username invalid — check the link",
            "ChannelPrivateError":       "Chat is private — you must be a member",
            "MessageIdInvalidError":     "Message ID invalid — check the link",
        }
        if n in M: return M[n]
        if n == "FloodWaitError":
            return f"Rate limit — wait {getattr(e,'seconds','?')}s"
        if "Connection" in n or "ConnectionReset" in n or "ConnectionError" in n or "OSError" in n:
            return "Connection to Telegram dropped. Retrying usually fixes this."
        if "Timeout" in n:
            return "Telegram timeout — try again in a moment"
        low = m.lower()
        if "quota" in low or "storagequotaexceeded" in low:
            return "Google Drive quota exceeded (15 GB free)"
        if "input/output error" in low or "errno 5" in low:
            return "Drive write failed — likely quota or stale mount. Re-run Setup."
        if "Cannot find any entity" in m:
            return "Chat not found — join it first"
        return f"{n}: {m[:140]}"

    def _safe(name):
        name = re.sub(r'[\\/:*?"<>|]+', "_", name).strip()
        return name or "file.bin"

    async def _get_dest_folder(msg):
        sender_name = "Unknown Sender"
        try:
            sender = msg.sender
            if not sender:
                sender = await msg.get_sender()

            if sender:
                if getattr(sender, 'title', None):
                    sender_name = sender.title
                elif getattr(sender, 'first_name', None):
                    sender_name = sender.first_name
                elif getattr(sender, 'username', None):
                    sender_name = sender.username
        except Exception:
            pass

        date_str = msg.date.strftime('%Y-%m-%d') if getattr(msg, 'date', None) else "Unknown Date"
        folder_name = re.sub(r'[\\/:*?"<>|]+', "_", f"{sender_name} - {date_str}").strip()
        return os.path.join(STATE["drive_root"], folder_name)

    async def _ensure_connected():
        client = STATE.get("tg")
        if not client:
            raise RuntimeError("Telegram not connected — run Step 2")
        if not client.is_connected():
            await client.connect()
            await asyncio.sleep(1.5)

    async def _tg_call(coro_factory, retries=4, delay=2.0):
        last_err = None
        for attempt in range(1, retries + 1):
            try:
                await _ensure_connected()
                return await coro_factory()
            except FloodWaitError:
                raise
            except (TimedOutError, ServerError, ConnectionError, OSError) as e:
                last_err = e
                if attempt < retries:
                    await asyncio.sleep(delay * attempt)
                    continue
                raise
            except Exception as e:
                last_err = e
                n = type(e).__name__
                if "Connection" in n or "Timeout" in n:
                    if attempt < retries:
                        await asyncio.sleep(delay * attempt)
                        continue
                raise
        if last_err: raise last_err

    async def _download(entity, msg_obj, path, on_progress=None):
        total = 0
        last_update = [0.0]

        async def cb(current, _total):
            nonlocal total
            total = _total
            if on_progress:
                now = time.time()
                if now - last_update[0] > 0.3 or current >= total:
                    on_progress(current, total)
                    last_update[0] = now

        async def _do():
            nonlocal msg_obj
            try:
                await msg_obj.download_media(file=path, progress_callback=cb)
            except Exception as e:
                err_str = str(e).lower()
                if "file reference" in err_str or "expired" in err_str or "connection" in err_str or "timeout" in err_str:
                    fresh_msg = await STATE["tg"].get_messages(entity, ids=msg_obj.id)
                    if fresh_msg and fresh_msg.file:
                        msg_obj = fresh_msg
                        await msg_obj.download_media(file=path, progress_callback=cb)
                    else:
                        raise
                else:
                    raise

        await _tg_call(_do, retries=4)

    async def _copy_to_drive(src, dst, on_progress=None):
        total = os.path.getsize(src)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        copied = 0
        last_update = 0.0
        with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
            while True:
                chunk = fsrc.read(2 * 1024 * 1024)
                if not chunk: break
                fdst.write(chunk)
                copied += len(chunk)
                if on_progress:
                    now = time.time()
                    if now - last_update > 0.3 or copied >= total:
                        on_progress(copied, total)
                        last_update = now
                await asyncio.sleep(0)
        shutil.copystat(src, dst)

    async def tg_request_code(api_id, api_hash, phone):
        if STATE["tg"]:
            try: await STATE["tg"].disconnect()
            except Exception: pass
        c = TelegramClient(StringSession(), api_id, api_hash)
        await c.connect()
        sent = await c.send_code_request(phone)
        c._code_hash = sent.phone_code_hash
        STATE["tg"] = c

    async def tg_verify_code(phone, code):
        try:
            await STATE["tg"].sign_in(phone, code=code, phone_code_hash=STATE["tg"]._code_hash)
            return True
        except SessionPasswordNeededError:
            return False

    async def tg_verify_2fa(password):
        await STATE["tg"].sign_in(password=password)
        return True

    async def tg_session_string():
        return STATE["tg"].session.save()

    async def tg_me():
        return await _tg_call(lambda: STATE["tg"].get_me())

    async def _resolve(link):
        m = _LINK_RE.match(link)
        if not m:
            raise ValueError("Not a valid Telegram message link")
        ref, msg_id = m.group(1), int(m.group(2))

        async def _do():
            entity = await STATE["tg"].get_entity(int(f"-100{ref}") if ref.isdigit() else ref)
            msg = await STATE["tg"].get_messages(entity, ids=msg_id)
            return entity, msg

        return await _tg_call(_do)

    async def _expand_album(entity, msg):
        if not msg.grouped_id:
            return [msg]
        lo, hi = max(1, msg.id - 10), msg.id + 10

        async def _fetch():
            return await STATE["tg"].get_messages(entity, ids=list(range(lo, hi + 1)))

        batch = await _tg_call(_fetch)
        return [x for x in batch if x and x.grouped_id == msg.grouped_id and x.file]

    async def _fetch_and_preview(link, album=True, resume=True):
        if not STATE.get("tg"):
            raise RuntimeError("Telegram not connected — run Step 2")

        entity, msg = await _resolve(link)
        if msg is None:
            raise RuntimeError("Message not found — deleted, or you're not a member")
        if not msg.file:
            raise RuntimeError("That message has no downloadable file")

        msgs = await _expand_album(entity, msg) if album else [msg]
        folder_path = await _get_dest_folder(msg)

        out = []
        for mm in msgs:
            fname = _safe(mm.file.name or f"tg_{mm.id}{mm.file.ext or '.bin'}")
            dst   = os.path.join(folder_path, fname)
            size  = mm.file.size or 0

            if resume and os.path.exists(dst) and os.path.getsize(dst) == size:
                action = "skip"
            elif os.path.exists(dst):
                action = "overwrite"
            else:
                action = "upload"

            out.append((mm.id, fname, size, action))
        return out

    async def _upload_one(link, selected_ids=None, resume=True, on_progress=None):
        if not STATE.get("tg"):
            raise RuntimeError("Telegram not connected — run Step 2")

        entity, msg = await _resolve(link)
        if msg is None:
            raise RuntimeError("Message not found — deleted, or you're not a member")
        if not msg.file:
            raise RuntimeError("That message has no downloadable file")

        msgs = await _expand_album(entity, msg)

        if selected_ids is not None:
            msgs = [m for m in msgs if m.id in selected_ids]
            if not msgs:
                raise RuntimeError("No files selected to upload.")

        folder_path = await _get_dest_folder(msg)

        total_files = len(msgs)
        results = {"uploaded": [], "skipped": [], "overwritten": [], "failed": []}

        for idx, mm in enumerate(msgs, 1):
            fname = _safe(mm.file.name or f"tg_{mm.id}{mm.file.ext or '.bin'}")
            tmp   = os.path.join(STATE["tmp_dir"], f"{mm.id}_{fname}")
            dst   = os.path.join(folder_path, fname)
            target_size = mm.file.size or 0

            if resume and os.path.exists(dst):
                if os.path.getsize(dst) == target_size:
                    results["skipped"].append(dst)
                    if on_progress:
                        on_progress(0, 0, "skip", fname, idx, total_files)
                    continue
                try: os.remove(dst)
                except OSError: pass

            overwriting = os.path.exists(dst)
            if overwriting:
                try: os.remove(dst)
                except OSError: pass

            try:
                if on_progress:
                    on_progress(0, target_size, "Downloading", fname, idx, total_files)
                await _download(
                    entity, mm, tmp,
                    on_progress=(lambda c, t: on_progress and on_progress(c, t, "Downloading", fname, idx, total_files)),
                )

                if on_progress:
                    on_progress(0, target_size, "Copying to Drive", fname, idx, total_files)
                await _copy_to_drive(
                    tmp, dst,
                    on_progress=(lambda c, t: on_progress and on_progress(c, t, "Copying to Drive", fname, idx, total_files)),
                )

                if overwriting:
                    results["overwritten"].append(dst)
                else:
                    results["uploaded"].append(dst)

            except Exception:
                results["failed"].append(fname)
                raise
            finally:
                try: os.remove(tmp)
                except OSError: pass

        return results

    display(_badge("ok", "✅ Backend ready."))

## 🔑 Step 2: Connect to Telegram

**First time?** You will need an `API ID` and `API Hash`.
1. Go to [my.telegram.org](https://my.telegram.org) and log in.
2. Click **API development tools**.
3. Fill out the form (anything works) to generate your ID and Hash.
4. Paste them here, enter your phone number (with country code, e.g., `+201234567890`), and click **1. Send code**.

Check your Telegram app for the login code, enter it, and click **2. Verify code**. If you have Two-Step Verification (2FA) enabled, enter your password and click **3. Submit 2FA**.

*Note: Your session is saved to your Google Drive. The next time you open this notebook, it will automatically reconnect, and you can skip this form entirely!*

In [ ]:
#@title 🔑 RUN: Connect to Telegram { display-mode: "form" }

if "STATE" not in globals() or "tg_request_code" not in globals():
    display(_badge("warn", "⚠️  Run Setup and Backend first."))
else:
    import asyncio, os, json
    section(2, "Connect Telegram")

    SESSION_FILE = os.path.join(STATE["creds_dir"], "tg_session.txt")
    API_FILE     = os.path.join(STATE["creds_dir"], "tg_api.json")
    os.makedirs(STATE["creds_dir"], exist_ok=True)

    restored = False
    if os.path.exists(SESSION_FILE) and os.path.exists(API_FILE):
        try:
            with open(SESSION_FILE) as f: _sess = f.read().strip()
            with open(API_FILE) as f:     _api = json.load(f)
            _c = TelegramClient(StringSession(_sess), _api["api_id"], _api["api_hash"])
            await _c.connect()
            if await _c.is_user_authorized():
                STATE["tg"] = _c
                STATE["tg_verified"] = True
                _me = await _c.get_me()
                display(_badge("ok",
                    f"✅ Reconnected as <b>{_me.first_name or _me.username}</b> "
                    f"(@{_me.username}). Skip to Step 3."))
                restored = True
            else:
                display(_badge("warn", "Saved session expired — log in below."))
        except Exception as e:
            display(_badge("warn", f"Auto-restore failed: {type(e).__name__}"))

    if not restored:
        STATE.pop("_tg_logged_in", None)

        api_id_in   = w.IntText(description="API ID", value=0, layout=w.Layout(width="380px"))
        api_hash_in = w.Password(description="API Hash", layout=w.Layout(width="380px"))
        phone_in    = w.Text(description="Phone", placeholder="+201234567890",
                             layout=w.Layout(width="380px"))
        code_in     = w.Text(description="Code", placeholder="code from Telegram",
                             layout=w.Layout(width="380px"))
        pw_in       = w.Password(description="2FA password",
                                 placeholder="only if 2FA enabled",
                                 layout=w.Layout(width="380px"))

        b_send   = w.Button(description="1. Send code",    button_style="primary",
                            layout=w.Layout(width="150px"))
        b_verify = w.Button(description="2. Verify code",  button_style="success",
                            layout=w.Layout(width="150px"))
        b_2fa    = w.Button(description="3. Submit 2FA",   button_style="warning",
                            layout=w.Layout(width="150px"))
        b_reset  = w.Button(description="Restart login",
                            layout=w.Layout(width="150px"))

        out_area = w.Output()

        display(w.VBox([
            w.HTML("<b>Telegram API credentials</b> — "
                   "<a href='https://my.telegram.org/apps' target='_blank'>"
                   "my.telegram.org/apps</a><br>"
                   "<span style='color:#888;font-size:12px'>"
                   "First time only — saved to Drive, skipped on future runs."
                   "</span>"),
            api_id_in, api_hash_in, phone_in, code_in, pw_in,
            w.HBox([b_send, b_verify, b_2fa, b_reset]),
            out_area,
        ]))

        def _final(kind, msg):
            out_area.clear_output(wait=True)
            with out_area: display(_badge(kind, msg))

        def _wait(msg):
            out_area.clear_output(wait=True)
            with out_area: display(_badge("info", f"⏳ {msg}"))

        def _run(coro):
            try:
                try: loop = asyncio.get_event_loop()
                except RuntimeError:
                    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
                loop.run_until_complete(coro)
            except Exception as e:
                _final("err", _friendly(e))

        async def _s_send():
            if api_id_in.value <= 0 or not api_hash_in.value.strip():
                _final("warn", "Fill in API ID and API Hash first."); return
            if not phone_in.value.strip().startswith("+"):
                _final("warn", "Phone must include country code (e.g. +201234567890)."); return
            await tg_request_code(api_id_in.value, api_hash_in.value.strip(),
                                  phone_in.value.strip())
            STATE["tg_verified"] = False
            STATE.pop("_tg_logged_in", None)
            _final("info",
                f"Code sent to <code>{phone_in.value}</code>. "
                "Enter it above, then click <b>2. Verify code</b>.")

        async def _s_verify():
            if not STATE.get("tg") or not hasattr(STATE["tg"], "_code_hash"):
                _final("warn", "Click <b>1. Send code</b> first."); return
            if not code_in.value.strip():
                _final("warn", "Enter the code Telegram sent you."); return
            ok = await tg_verify_code(phone_in.value.strip(), code_in.value.strip())
            STATE["tg_verified"] = True
            if not ok:
                _final("info",
                    "Code accepted. <b>2FA enabled</b> — enter your cloud "
                    "password above, then click <b>3. Submit 2FA</b>.")
                return
            await _s_finish()

        async def _s_2fa():
            if not STATE.get("tg"):
                _final("warn", "Click <b>1. Send code</b> first."); return
            if not STATE.get("tg_verified"):
                _final("warn", "Verify the code first — click <b>2. Verify code</b>."); return
            if not pw_in.value:
                _final("warn", "Enter your Telegram cloud password."); return
            if not STATE["tg"].is_connected():
                await STATE["tg"].connect()
            await tg_verify_2fa(pw_in.value)
            await _s_finish()

        async def _s_finish():
            me = await tg_me()
            STATE["_tg_logged_in"] = True
            sess = await tg_session_string()

            try:
                os.makedirs(STATE["creds_dir"], exist_ok=True)
                with open(SESSION_FILE, "w") as f: f.write(sess)
                os.chmod(SESSION_FILE, 0o600)
                with open(API_FILE, "w") as f:
                    json.dump({"api_id": api_id_in.value,
                               "api_hash": api_hash_in.value.strip()}, f)
                os.chmod(API_FILE, 0o600)
                note = "Session saved to Drive."
            except Exception as e:
                note = f"Could not save session ({type(e).__name__})."

            for btn in (b_send, b_verify, b_2fa):
                btn.disabled = True

            _final("ok",
                f"✅ Logged in as <b>{me.first_name or me.username}</b>"
                + (f" (@{me.username})" if me.username else "")
                + f". {note}")

        def _click_send(b):
            _wait("Please wait… requesting login code")
            _run(_s_send())

        def _click_verify(b):
            _wait("Please wait… verifying code")
            _run(_s_verify())

        def _click_2fa(b):
            _wait("Please wait… submitting 2FA password")
            _run(_s_2fa())

        def _reset(b):
            try:
                if STATE.get("tg"):
                    try:
                        loop = asyncio.get_event_loop()
                        loop.run_until_complete(STATE["tg"].disconnect())
                    except Exception: pass
                STATE["tg"] = None
                STATE["tg_verified"] = False
                STATE.pop("_tg_logged_in", None)
                code_in.value = ""; pw_in.value = ""
                for btn in (b_send, b_verify, b_2fa):
                    btn.disabled = False
                _final("info", "🔄 Reset. Start with <b>1. Send code</b>.")
            except Exception as e:
                _final("err", f"Reset error: {type(e).__name__}: {e}")

        b_send.on_click(_click_send)
        b_verify.on_click(_click_verify)
        b_2fa.on_click(_click_2fa)
        b_reset.on_click(_reset)

## 📤 Step 3: Download & Upload

1. Paste a Telegram message link (e.g., `https://t.me/channel/123` or a private group link).
2. Click **🔍 Fetch & Preview** to fetch the file details.
3. A list of files will appear below. Uncheck any files you *don't* want to download.
4. Click **⬆️ Upload**.

*You will see a live progress bar, download/upload speed, time elapsed, and estimated time remaining inside the widget.*

In [ ]:
#@title 📤 RUN: Download & Upload { display-mode: "form" }

if "STATE" not in globals() or "_upload_one" not in globals():
    display(_badge("warn", "⚠️  Run Setup and Backend first."))
else:
    import asyncio, os, time, math
    section(3, "Download & Upload")

    link_in    = w.Text(description="Link",
                        placeholder="https://t.me/channel/12345",
                        layout=w.Layout(width="100%"),
                        style={"description_width": "60px"})
    album_chk  = w.Checkbox(value=True, description="Include album items", indent=False)
    resume_chk = w.Checkbox(value=True, description="Resume (skip existing)", indent=False)

    fetch_btn   = w.Button(description="🔍 Fetch & Preview", layout=w.Layout(width="160px"))
    up_btn      = w.Button(description="⬆️  Upload", button_style="success",
                           layout=w.Layout(width="140px"))

    select_all_btn = w.Button(description="☑️ Select All", layout=w.Layout(width="130px", display="none"))

    files_area  = w.VBox([])
    out_html    = w.HTML()

    display(w.VBox([
        w.HTML("<b>Paste a Telegram message link</b> — "
               "<code>t.me/user/123</code> · "
               "<code>t.me/c/1234567890/45</code>"),
        link_in,
        w.HBox([album_chk, resume_chk]),
        w.HBox([fetch_btn, up_btn, select_all_btn]),
        out_html,
        files_area,
    ]))

    _checkbox_map = {}
    _speed_state = {"last_time": 0.0, "last_bytes": 0, "speed_bps": 0, "phase_key": "", "phase_start": 0.0}

    def _render_badge(kind, text):
        out_html.value = f"<div class='badge {kind}'>{text}</div>"
        print(f"[{kind}] {text.replace('<br>', ' | ')[:100]}...", end="\r")

    def _format_speed(bps):
        if bps < 1024:       return f"{bps:.0f} B/s"
        if bps < 1024**2:    return f"{bps/1024:.1f} KB/s"
        return f"{bps/1024**2:.1f} MB/s"

    def _format_time(seconds):
        if seconds is None or seconds < 0 or math.isinf(seconds) or math.isnan(seconds):
            return "--:--"
        m, s = divmod(int(seconds), 60)
        h, m = divmod(m, 60)
        if h > 0:
            return f"{h:02d}:{m:02d}:{s:02d}"
        return f"{m:02d}:{s:02d}"

    def _run(coro):
        try:
            try: loop = asyncio.get_event_loop()
            except RuntimeError:
                loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
            loop.run_until_complete(coro)
        except Exception as e:
            _render_badge("err", _friendly(e))

    def _validate():
        link = link_in.value.strip()
        if not link:
            _render_badge("warn", "Paste a Telegram link first.")
            return None
        if not re.match(r"https?://t\.me/(?:c/)?([^/?]+)(?:/\d+)?/(\d+)", link):
            _render_badge("err", "Not a valid Telegram link.")
            return None
        if not STATE.get("tg"):
            _render_badge("err", "Telegram not connected — run Step 2.")
            return None
        if not os.path.exists(STATE["drive_root"]):
            try: os.makedirs(STATE["drive_root"], exist_ok=True)
            except Exception:
                _render_badge("err", f"Cannot create <code>{STATE['drive_root']}</code>.")
                return None
        return link

    def _on_select_all(b):
        if "Select" in select_all_btn.description:
            for cb in _checkbox_map.keys(): cb.value = True
            select_all_btn.description = "⬜ Unselect All"
        else:
            for cb in _checkbox_map.keys(): cb.value = False
            select_all_btn.description = "☑️ Select All"

    def _on_fetch(b):
        link = _validate()
        if not link: return

        files_area.children = ()
        _checkbox_map.clear()
        select_all_btn.layout.display = "none"
        _render_badge("info", "⏳ Please wait… fetching files")

        async def _go():
            try:
                files = await _fetch_and_preview(link, album=album_chk.value, resume=resume_chk.value)
                if not files:
                    _render_badge("warn", "No files found in link.")
                    return

                new_checkboxes = []
                total_mb = 0
                for msg_id, fname, size, action in files:
                    if action != "skip": total_mb += size

                    action_icon = {"upload": "⬆️", "overwrite": "🔁", "skip": "⏩"}[action]
                    label = f"{action_icon} {fname} ({size/1e6:.1f} MB)"

                    cb = w.Checkbox(value=(action != "skip"), description=label, indent=False, layout=w.Layout(width="100%"))
                    _checkbox_map[cb] = msg_id
                    new_checkboxes.append(cb)

                files_area.children = [
                    w.HTML(f"<b>Files to upload:</b> {total_mb/1e6:.1f} MB total<br>"
                           "<span style='font-size:11px;color:#888'>⬆️ upload · 🔁 overwrite · ⏩ skip</span>")
                ] + new_checkboxes

                select_all_btn.description = "☑️ Select All"
                select_all_btn.layout.display = ""
                _render_badge("ok", "✅ Ready. Adjust selection below, then click Upload.")
            except Exception as e:
                _render_badge("err", _friendly(e))

        _run(_go())

    def _render_progress(current, total, phase, fname, idx, total_files):
        if phase == "skip":
            _render_badge("info", f"⏩ Already in Drive — <code>{fname}</code>")
            return

        now = time.time()
        phase_key = f"{phase}_{fname}"

        if _speed_state["phase_key"] != phase_key:
            _speed_state["phase_key"] = phase_key
            _speed_state["phase_start"] = now
            _speed_state["last_time"] = 0
            _speed_state["last_bytes"] = 0

        if _speed_state["last_time"] == 0 or current < _speed_state["last_bytes"]:
            _speed_state["last_time"] = now
            _speed_state["last_bytes"] = current
        time_diff  = now - _speed_state["last_time"]
        bytes_diff = current - _speed_state["last_bytes"]
        if time_diff > 0.5:
            _speed_state["speed_bps"] = bytes_diff / time_diff
            _speed_state["last_time"] = now
            _speed_state["last_bytes"] = current

        speed_str = _format_speed(_speed_state["speed_bps"])

        elapsed = now - _speed_state["phase_start"]
        if _speed_state["speed_bps"] > 0 and total > current:
            eta = (total - current) / _speed_state["speed_bps"]
        else:
            eta = -1

        time_str = f"⏱️ {_format_time(elapsed)} elapsed · {_format_time(eta)} left"

        pct = int(current / total * 100) if total else 0
        filled = int(pct / 5)
        bar = "█" * filled + "░" * (20 - filled)
        file_info = f"File {idx}/{total_files} · " if total_files > 1 else ""

        _render_badge("info",
            f"⏳ {phase}… {bar} {pct}%<br>"
            f"{file_info}<code>{fname}</code> · "
            f"{current/1e6:.1f} / {total/1e6:.1f} MB · <b>{speed_str}</b><br>"
            f"<span style='font-size:12px;color:#555'>{time_str}</span>")

    def _on_upload(b):
        link = _validate()
        if not link: return

        selected_ids = None
        if _checkbox_map:
            selected_ids = [mid for cb, mid in _checkbox_map.items() if cb.value]
            if not selected_ids:
                _render_badge("warn", "No files selected.")
                return

        files_area.children = ()
        _checkbox_map.clear()
        select_all_btn.layout.display = "none"
        resume = resume_chk.value

        up_btn.disabled = True
        fetch_btn.disabled = True
        _render_badge("info", "⏳ Please wait…")

        async def _go():
            try:
                results = await _upload_one(
                    link, selected_ids=selected_ids, resume=resume,
                    on_progress=_render_progress)

                up = len(results["uploaded"])
                sk = len(results["skipped"])
                ov = len(results["overwritten"])
                fl = len(results["failed"])

                parts = []
                if up: parts.append(f"<b>{up}</b> uploaded")
                if ov: parts.append(f"<b>{ov}</b> overwritten")
                if sk: parts.append(f"<b>{sk}</b> skipped")
                if fl: parts.append(f"<b>{fl}</b> failed")

                lines = ""
                if results["uploaded"]:
                    lines += "<br><br><b>New:</b><br>" + "<br>".join(f"<code>{p}</code>" for p in results["uploaded"])
                if results["overwritten"]:
                    lines += "<br><br><b>Overwritten:</b><br>" + "<br>".join(f"<code>{p}</code>" for p in results["overwritten"])

                if up == 0 and ov == 0 and sk > 0:
                    _render_badge("info", f"⏩ All <b>{sk}</b> file(s) already in Drive — nothing to do.")
                elif fl > 0:
                    _render_badge("warn", "⚠️  " + " · ".join(parts) + lines)
                else:
                    _render_badge("ok", "✅  Done — " + " · ".join(parts) + lines)

            except Exception as e:
                _render_badge("err", _friendly(e))

            finally:
                up_btn.disabled = False
                fetch_btn.disabled = False

        _run(_go())

    fetch_btn.on_click(_on_fetch)
    up_btn.on_click(_on_upload)
    select_all_btn.on_click(_on_select_all)

## 📊 Step 4: Live Dashboard

This dashboard updates automatically every 3 seconds. You can monitor your total storage used, the exact dates the files were downloaded, and the recent history of files added to your Drive.

*(Note: You must run this cell at least once to start the auto-refresh timer).*

In [ ]:
#@title 📊 RUN: Live Dashboard { display-mode: "form" }

if "STATE" not in globals():
    display(_badge("warn", "⚠️  Run Setup first."))
else:
    import os, time
    from google.colab import output
    from IPython.display import Javascript, display
    import ipywidgets as w

    section(4, "Live Dashboard")

    dash_html = w.HTML()
    ref_btn = w.Button(description="🔄 Refresh Now", button_style="primary", layout=w.Layout(width="130px"))

    def _render():
        try:
            root = STATE.get("drive_root", "/content/drive/MyDrive/TelegramToGDrive Downloads")
            tg_ok = STATE.get("tg") is not None

            files = []
            if os.path.isdir(root):
                for dirpath, _, filenames in os.walk(root):
                    for fn in filenames:
                        full = os.path.join(dirpath, fn)
                        if os.path.isfile(full):
                            size = os.path.getsize(full)
                            mtime = os.path.getmtime(full)
                            date_str = time.strftime('%Y-%m-%d %H:%M', time.localtime(mtime))
                            rel_path = os.path.relpath(full, root)
                            files.append((rel_path, size, date_str, mtime))

            files.sort(key=lambda x: x[3], reverse=True)
            total = sum(s for _, s, _, _ in files)
            current_time = time.strftime("%H:%M:%S")

            rows = ""
            for fn, sz, date_str, _ in files[:15]:
                rows += (
                    "<tr>"
                    f"<td style='padding:8px 12px;border-bottom:1px solid #eee;"
                    f"font-size:12px'><code>{fn}</code></td>"
                    f"<td style='padding:8px 12px;border-bottom:1px solid #eee;"
                    f"font-size:12px;color:#666'>{date_str}</td>"
                    f"<td style='padding:8px 12px;border-bottom:1px solid #eee;"
                    f"font-size:12px;text-align:right'>{sz/1e6:.1f} MB</td>"
                    "</tr>")

            if not rows:
                rows = ("<tr><td colspan='3' style='padding:16px;text-align:center;"
                        "color:#888;font-size:13px'>No files yet.</td></tr>")

            html = f"""
            <div style='background:#fafafa;border:1px solid #e0e0e0;border-radius:10px;
                        padding:18px;font-family:-apple-system,system-ui,sans-serif'>
              <div style='display:grid;grid-template-columns:1fr 1fr 1fr;
                          gap:12px;margin-bottom:16px'>
                <div style='background:#fff;border:1px solid #eee;border-radius:8px;padding:14px'>
                  <div style='font-size:11px;color:#888;text-transform:uppercase;
                              letter-spacing:1px'>Telegram</div>
                  <div style='font-size:15px;font-weight:600;margin-top:4px'>
                    {'✅ connected' if tg_ok else '❌ not connected'}</div>
                </div>
                <div style='background:#fff;border:1px solid #eee;border-radius:8px;padding:14px'>
                  <div style='font-size:11px;color:#888;text-transform:uppercase;
                              letter-spacing:1px'>Files in Drive</div>
                  <div style='font-size:15px;font-weight:600;margin-top:4px'>{len(files)}</div>
                </div>
                <div style='background:#fff;border:1px solid #eee;border-radius:8px;padding:14px'>
                  <div style='font-size:11px;color:#888;text-transform:uppercase;
                              letter-spacing:1px'>Total size</div>
                  <div style='font-size:15px;font-weight:600;margin-top:4px'>{total/1e9:.2f} GB</div>
                </div>
              </div>
              <div style='font-size:13px;font-weight:600;margin-bottom:8px;display:flex;justify-content:space-between'>
                <span>Recent files in <code>TelegramToGDrive Downloads</code></span>
                <span style='color:#888;font-weight:normal;font-size:11px'>Updated: {current_time}</span>
              </div>
              <table style='width:100%;border-collapse:collapse;background:#fff;
                            border-radius:8px;overflow:hidden;border:1px solid #eee'>
                <thead><tr style='background:#f5f5f5'>
                  <th style='padding:8px 12px;text-align:left;font-size:11px;color:#666'>FILE PATH</th>
                  <th style='padding:8px 12px;text-align:left;font-size:11px;color:#666'>DATE MODIFIED</th>
                  <th style='padding:8px 12px;text-align:right;font-size:11px;color:#666'>SIZE</th>
                </tr></thead><tbody>{rows}</tbody>
              </table>
            </div>"""

            dash_html.value = html
        except Exception as e:
            dash_html.value = f"<div style='color:red'>Dashboard Error: {e}</div>"

    output.register_callback('dash_refresh_cb', _render)

    ref_btn.on_click(lambda b: _render())

    display(w.VBox([w.HBox([ref_btn]), dash_html]))

    display(Javascript("""
    if (window.dashAutoRefresh) clearInterval(window.dashAutoRefresh);
    window.dashAutoRefresh = setInterval(function() {
        google.colab.kernel.invokeFunction('dash_refresh_cb', [], {});
    }, 3000);
    """))

    _render()

## 🗑 Step 5: Logout & Credentials

Finished for the day? You can safely close the tab and leave this notebook. It will remember your session next time.

If you ever need to log out completely (or switch accounts), click the button below. This will terminate the session on Telegram's servers and delete both your session and API credentials from your Google Drive.

*(Note: You will need to re-enter your API ID and Hash next time).*

In [ ]:
#@title 🗑 RUN: Logout & Credentials { display-mode: "form" }

if "STATE" not in globals():
    display(_badge("warn", "⚠️  Run Setup first."))
else:
    import os, asyncio
    section(5, "Remove saved credentials")

    SESSION_FILE = os.path.join(STATE["creds_dir"], "tg_session.txt")
    API_FILE     = os.path.join(STATE["creds_dir"], "tg_api.json")

    def _get_status_html():
        sess_exists = os.path.exists(SESSION_FILE)
        api_exists = os.path.exists(API_FILE)
        tg_connected = STATE.get("tg") is not None

        def make_row(title, is_ok, ok_text, fail_text):
            icon = "✅" if is_ok else "❌"
            color = "#2e7d32" if is_ok else "#c62828"
            text = ok_text if is_ok else fail_text
            return f"""
            <div style='display:flex; justify-content:space-between; align-items:center;
                        padding:12px 16px; border-bottom:1px solid #f0f0f0;'>
                <span style='color:#444; font-weight:500;'>{title}</span>
                <span style='color:{color}; font-weight:600; font-size:13px;'>{icon} {text}</span>
            </div>
            """

        html = "<div style='background:#fff; border:1px solid #e0e0e0; border-radius:10px; overflow:hidden; font-family:-apple-system,system-ui,sans-serif; font-size:13px;'>"
        html += make_row("Telegram Session", sess_exists, "Saved to Drive", "Not found")
        html += make_row("API Credentials", api_exists, "Saved to Drive", "Not found")
        html += make_row("Connection Status", tg_connected, "Connected", "Disconnected")

        # Remove the bottom border from the last row
        html = html.replace('border-bottom:1px solid #f0f0f0;\'>\n                <span style=\'color:#444; font-weight:500;\'>Connection Status', "'>\n                <span style='color:#444; font-weight:500;'>Connection Status")
        html += "</div>"
        return html

    status_out = w.Output()
    def _refresh_status():
        status_out.clear_output(wait=True)
        with status_out:
            display(HTML(_get_status_html()))
    _refresh_status()

    confirm = w.Checkbox(value=False,
        description="I understand — this logs me out and deletes my API credentials",
        indent=False, layout=w.Layout(width="100%"))
    del_btn = w.Button(description="🗑  Log out & delete all credentials",
                       button_style="danger", disabled=True,
                       layout=w.Layout(width="300px"))
    result_out = w.Output()

    display(w.VBox([
        w.HTML("<b>Removes:</b> the Telegram session AND your API ID/Hash from "
               "<code>TelegramToGDrive Credentials/</code>, terminates the session on "
               "Telegram's servers, and disconnects.<br>"
               "<span style='color:#888;font-size:12px'>"
               "⚠️ You will need to enter your API ID / hash and log in from scratch next time."
               "</span>"),
        status_out,
        w.HTML("<hr style='margin:10px 0;border:none;border-top:1px solid #eee'>"),
        confirm,
        del_btn,
        result_out,
    ]))

    def _gate(*_):
        del_btn.disabled = not confirm.value
    confirm.observe(_gate, names="value")

    def _do_delete(b):
        result_out.clear_output(wait=True)
        with result_out:
            display(_badge("info", "⏳ Please wait… logging out and deleting files"))

        remote_ok = False
        if STATE.get("tg"):
            try:
                async def _logout(): await STATE["tg"].log_out()
                try: loop = asyncio.get_event_loop()
                except RuntimeError:
                    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
                loop.run_until_complete(_logout())
                remote_ok = True
            except Exception:
                pass

        # Delete files
        session_deleted = False
        api_deleted = False
        try:
            if os.path.exists(SESSION_FILE):
                os.remove(SESSION_FILE)
                session_deleted = True
            if os.path.exists(API_FILE):
                os.remove(API_FILE)
                api_deleted = True
        except Exception:
            pass

        # Disconnect memory
        try:
            if STATE.get("tg"):
                async def _dc():
                    try: await STATE["tg"].disconnect()
                    except Exception: pass
                try:
                    loop = asyncio.get_event_loop()
                    loop.run_until_complete(_dc())
                except Exception: pass
            STATE["tg"] = None
            STATE["tg_verified"] = False
            STATE.pop("_tg_logged_in", None)
        except Exception:
            pass

        # Result
        result_out.clear_output(wait=True)
        with result_out:
            if remote_ok and session_deleted and api_deleted:
                display(_badge("ok",
                    "✅ Fully logged out. Session terminated on Telegram's servers "
                    "and all credentials deleted from Drive."))
            elif session_deleted or api_deleted:
                display(_badge("warn",
                    "Files deleted from Drive, but couldn't reach Telegram to "
                    "terminate the session. Revoke manually at "
                    "<a href='https://web.telegram.org/k/#?settings/devices' "
                    "target='_blank'>Telegram → Devices</a>."))
            else:
                display(_badge("info",
                    "Nothing was deleted — credentials weren't present."))

            display(_badge("info",
                "Run <b>🔑 Step 2</b> to log in and set up new API credentials whenever you're ready."))

        confirm.value = False
        del_btn.disabled = True
        _refresh_status()

    del_btn.on_click(_do_delete)